# 01. Оценка ответов LLM

Небольшая рубрика для слепого сравнения моделей без специальных библиотек.

Стек: Python standard library. Выполняйте ячейки сверху вниз.

In [ ]:
from statistics import mean

rubric = {
    "точность": 0.4,
    "полнота": 0.3,
    "формат": 0.2,
    "стиль": 0.1,
}

checks = [
    {
        "case": "Возврат товара",
        "model_A": {"точность": 5, "полнота": 4, "формат": 5, "стиль": 4},
        "model_B": {"точность": 3, "полнота": 5, "формат": 4, "стиль": 5},
    },
    {
        "case": "Изменение адреса",
        "model_A": {"точность": 4, "полнота": 4, "формат": 5, "стиль": 4},
        "model_B": {"точность": 4, "полнота": 3, "формат": 4, "стиль": 5},
    },
    {
        "case": "Нет данных в базе",
        "model_A": {"точность": 5, "полнота": 3, "формат": 5, "стиль": 4},
        "model_B": {"точность": 1, "полнота": 5, "формат": 4, "стиль": 5},
    },
]

def weighted_score(scores):
    return sum(
        scores[criterion] * weight
        for criterion, weight in rubric.items()
    )

results = {"model_A": [], "model_B": []}

for check in checks:
    print("\nКейс:", check["case"])

    for model in results:
        score = weighted_score(check[model])
        results[model].append(score)
        print(model, round(score, 2))

print("\nСредний балл")

for model, scores in results.items():
    print(model, round(mean(scores), 2))

Замените оценки результатами слепой проверки 15–30 рабочих примеров. Средний балл дополняйте перечнем критичных ошибок.

In [ ]:
import json
import urllib.request

MODEL = "qwen2.5:3b"

def ask_local_model(prompt):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.8},
    }

    request = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        return json.load(response)["response"]

prompt = """
Подготовь короткий ответ клиенту о возврате товара.
Точные правила возврата не предоставлены.
Не придумывай сроки и условия.
"""

for attempt in range(3):
    print(f"\nОтвет {attempt + 1}")
    print(ask_local_model(prompt))